<a href="https://colab.research.google.com/github/Anchor-head/ScienceGenreClassification/blob/main/SetFit-SciBERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip checkpoints_archive.zip

Archive:  checkpoints_archive.zip
   creating: checkpoint-876/
   creating: checkpoint-876/1_Pooling/
  inflating: checkpoint-876/README.md  
  inflating: checkpoint-876/config.json  
  inflating: checkpoint-876/optimizer.pt  
  inflating: checkpoint-876/rng_state.pth  
  inflating: checkpoint-876/config_sentence_transformers.json  
  inflating: checkpoint-876/vocab.txt  
  inflating: checkpoint-876/tokenizer_config.json  
  inflating: checkpoint-876/training_args.bin  
  inflating: checkpoint-876/modules.json  
  inflating: checkpoint-876/model.safetensors  
  inflating: checkpoint-876/sentence_bert_config.json  
  inflating: checkpoint-876/scheduler.pt  
  inflating: checkpoint-876/special_tokens_map.json  
  inflating: checkpoint-876/tokenizer.json  
  inflating: checkpoint-876/trainer_state.json  
  inflating: checkpoint-876/1_Pooling/config.json  


In [1]:
!pip install "datasets>=2.15.0,<3.0.0"
!pip install setfit
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 16.8 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.6.1 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; pl

# First model
Model: scincl

Train-test split: 1

Batch size: 16

Epochs: 4

Accuracy: 0.48905

In [ ]:
from datasets import load_dataset, Dataset
from setfit import SetFitModel, Trainer, TrainingArguments, sample_dataset


df = load_dataset("parquet", data_files={'train': "train_split.parquet", 'test': "test_split.parquet"})

# Load a dataset from the Hugging Face Hub
'''
train_df = load_dataset("parquet", data_files="train_split.parquet")
train_text = train_df["train"]["fullText"]
train_label = train_df["train"]["Document_subtype"]
train_data = Dataset.from_dict({ "fullText" : train_text, "Document_subtype" : train_label })

test_df = load_dataset("parquet", data_files="test_split.parquet")
test_text = test_df["train"]["fullText"]
test_label = test_df["train"]["Document_subtype"]
test_data = Dataset.from_dict({ "fullText" : test_text, "Document_subtype" : test_label })

# Simulate the few-shot regime by sampling 8 examples per class
train_dataset = sample_dataset(dataset["train"], label_column="Document_subtype")
eval_dataset = dataset["validation"].select(range(100))
test_dataset = dataset["validation"].select(range(100, len(dataset["validation"])))
'''

# Load a SetFit model from Hub
model = SetFitModel.from_pretrained(
    "malteos/scincl",
    labels=["Retraction", "Correction", "Editorial", "Policy, Science and Society", "News Journalism", "Research Document", "Letter", "Book Review", "Societies and People", "Miscellanea"],
)

args = TrainingArguments(
    batch_size=16,
    num_epochs=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=df['train'],
    eval_dataset=df['test'],
    metric="accuracy",
    column_mapping={"fullText": "text", "Document_subtype": "label"}  # Map dataset columns to text/label expected by trainer
)

# Train and evaluate
trainer.train()
metrics = trainer.evaluate(df['test'])
print(metrics)
# {'accuracy': 0.8691709844559585}

'''
# Push model to the Hub
trainer.push_to_hub("tomaarsen/setfit-paraphrase-mpnet-base-v2-sst2")

# Download from Hub
model = SetFitModel.from_pretrained("tomaarsen/setfit-paraphrase-mpnet-base-v2-sst2")
# Run inference
preds = model.predict(["i loved the spiderman movie!", "pineapple on pizza is the worst 🤮"])
print(preds)
# ["positive", "negative"]
'''

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Applying column mapping to the training dataset
Applying column mapping to the evaluation dataset


AttributeError: 'Column' object has no attribute 'sort'

# Second model
Model: scincl

Train-test split: 1

Batch size: 16

Epochs: 5

Accuracy: 0.49635

In [ ]:
from datasets import load_dataset, Dataset
from setfit import SetFitModel, Trainer, TrainingArguments, sample_dataset


df = load_dataset("parquet", data_files={'train': "train_split.parquet", 'test': "test_split.parquet"})

# Load a SetFit model from Hub
model = SetFitModel.from_pretrained(
    "malteos/scincl",
    labels=["Retraction", "Correction", "Editorial", "Policy, Science and Society", "News Journalism", "Research Document", "Letter", "Book Review", "Societies and People", "Miscellanea"],
)

args = TrainingArguments(
    batch_size=16,
    num_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=df['train'],
    eval_dataset=df['test'],
    metric="accuracy",
    column_mapping={"fullText": "text", "Document_subtype": "label"}  # Map dataset columns to text/label expected by trainer
)

# Train and evaluate
trainer.train()
metrics = trainer.evaluate(df['test'])
print(metrics)
# {'accuracy': 0.8691709844559585}

model.save_pretrained("./model2")

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Applying column mapping to the training dataset
Applying column mapping to the evaluation dataset
***** Running training *****
  Num unique pairs = 3502
  Batch size = 16
  Num epochs = 5


Epoch,Training Loss,Validation Loss
1,0.015900,0.266612
2,0.002600,0.250628
3,0.001900,0.249154
4,0.001600,0.248366
5,0.001400,0.248920


Applying column mapping to the evaluation dataset
***** Running evaluation *****


{'accuracy': 0.49635036496350365}


# Third model
Model: jordyvl

Train-test split: 1

Batch size: 16

Epochs: 5

Accuracy: 0.5401459854014599

In [ ]:
from datasets import load_dataset, Dataset
from setfit import SetFitModel, Trainer, TrainingArguments, sample_dataset


df = load_dataset("parquet", data_files={'train': "train_split.parquet", 'test': "test_split.parquet"})

# Load a SetFit model from Hub
model = SetFitModel.from_pretrained(
    "jordyvl/scibert_scivocab_uncased_sentence_transformer",
    labels=["Retraction", "Correction", "Editorial", "Policy, Science and Society", "News Journalism", "Research Document", "Letter", "Book Review", "Societies and People", "Miscellanea"],
)

args = TrainingArguments(
    batch_size=16,
    num_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=df['train'],
    eval_dataset=df['test'],
    metric="accuracy",
    column_mapping={"fullText": "text", "Document_subtype": "label"}  # Map dataset columns to text/label expected by trainer
)

# Train and evaluate
trainer.train()
metrics = trainer.evaluate(df['test'])
print(metrics)
# {'accuracy': 0.8691709844559585}

model.save_pretrained("./model3")

config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Applying column mapping to the training dataset
Applying column mapping to the evaluation dataset
***** Running training *****
  Num unique pairs = 3502
  Batch size = 16
  Num epochs = 5


Epoch,Training Loss,Validation Loss
1,0.018000,0.250013
2,0.000700,0.264815
3,0.000100,0.262812
4,0.000000,0.263298
5,0.000000,0.264023


Applying column mapping to the evaluation dataset
***** Running evaluation *****


{'accuracy': 0.5401459854014599}


# Fourth model
Model: jordyvl

Train-test split: 2

Batch size: 16

Epochs: 1

Accuracy: 0.5876

In [ ]:
from datasets import load_dataset, Dataset
from setfit import SetFitModel, Trainer, TrainingArguments, sample_dataset


df = load_dataset("parquet", data_files={'train': "train_split2.parquet", 'test': "test_split2.parquet"})

# Load a SetFit model from Hub
model = SetFitModel.from_pretrained(
    "jordyvl/scibert_scivocab_uncased_sentence_transformer",
    labels=["Retraction", "Correction", "Editorial", "Policy, Science and Society", "News Journalism", "Research Document", "Letter", "Book Review", "Societies and People", "Miscellanea"],
)

args = TrainingArguments(
    batch_size=16,
    num_epochs=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=df['train'],
    eval_dataset=df['test'],
    metric="accuracy",
    column_mapping={"fullText": "text", "Document_subtype": "label"}  # Map dataset columns to text/label expected by trainer
)

# Train and evaluate
trainer.train()
metrics = trainer.evaluate(df['test'])
print(metrics)
# {'accuracy': 0.8691709844559585}

model.save_pretrained("./model4")

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

...

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vamphills (vamphills-universit-du-qu-bec-montr-al) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
/usr/local/lib/python3.11/dist-packages/wandb/analytics/sentry.py:258: DeprecationWarning: The `Scope.user` setter is deprecated in favor of `Scope.set_user()`.
  self.scope.user = {"email": email}


Epoch,Training Loss,Validation Loss
1,0.004900,0.206104


Applying column mapping to the evaluation dataset
***** Running evaluation *****


{'accuracy': 0.5876288659793815}


# Fifth model
Model: jordyvl

Train-test split: 3

Batch size: 16

Epochs: 1

Accuracy: 0.5593

In [ ]:
from datasets import load_dataset, Dataset
from setfit import SetFitModel, Trainer, TrainingArguments, sample_dataset


df = load_dataset("parquet", data_files={'train': "train_split3.parquet", 'test': "test_split3.parquet"})

# Load a SetFit model from Hub
model = SetFitModel.from_pretrained(
    "jordyvl/scibert_scivocab_uncased_sentence_transformer",
    labels=["Retraction", "Correction", "Editorial", "Policy, Science and Society", "News Journalism", "Research Document", "Letter", "Book Review", "Societies and People", "Miscellanea"],
)

args = TrainingArguments(
    batch_size=16,
    num_epochs=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=df['train'],
    eval_dataset=df['test'],
    metric="accuracy",
    column_mapping={"fullText": "text", "Document_subtype": "label"}  # Map dataset columns to text/label expected by trainer
)

# Train and evaluate
trainer.train()
metrics = trainer.evaluate(df['test'])
print(metrics)
# {'accuracy': 0.8691709844559585}

model.save_pretrained("./model5")

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Applying column mapping to the training dataset
Applying column mapping to the evaluation dataset


Map:   0%|          | 0/141 [00:00<?, ? examples/s]

***** Running training *****
  Num unique pairs = 15718
  Batch size = 16
  Num epochs = 1


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
1,0.007400,0.187275


Applying column mapping to the evaluation dataset
***** Running evaluation *****


{'accuracy': 0.559322033898305}


# Sixth model

Model: jordyvl

Train-test split: 2

Batch size: 16

Epochs: 5

Accuracy: 0.597-0.6392

Accuracy at epoch 2: 0.56




In [3]:
from datasets import load_dataset, Dataset
from setfit import SetFitModel, Trainer, TrainingArguments, sample_dataset


df = load_dataset("parquet", data_files={'train': "train_split2.parquet", 'test': "test_split2.parquet"})

# Load a SetFit model from Hub
model = SetFitModel.from_pretrained("jordyvl/scibert_scivocab_uncased_sentence_transformer", labels=["Retraction", "Correction", "Editorial", "Policy, Science and Society", "News Journalism", "Research Document", "Letter", "Book Review", "Societies and People", "Miscellanea"])

args = TrainingArguments(
    batch_size=16,
    num_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=df['train'],
    eval_dataset=df['test'],
    metric="accuracy",
    column_mapping={"fullText": "text", "Document_subtype": "label"}  # Map dataset columns to text/label expected by trainer
)

# Train and evaluate
trainer.train()
metrics = trainer.evaluate(df['test'])
print(metrics)
# {'accuracy': 0.8691709844559585}

model.save_pretrained("./model6")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vamphills (vamphills-universit-du-qu-bec-montr-al) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
/usr/local/lib/python3.11/dist-packages/wandb/analytics/sentry.py:258: DeprecationWarning: The `Scope.user` setter is deprecated in favor of `Scope.set_user()`.
  self.scope.user = {"email": email}


Epoch,Training Loss,Validation Loss
1,0.007400,0.232288
2,0.000300,0.222064
3,0.000000,0.223389
4,0.000000,0.224551
5,0.000000,0.224349


Applying column mapping to the evaluation dataset
***** Running evaluation *****


{'accuracy': 0.5979381443298969}


# Other

In [4]:
import shutil
shutil.make_archive('model6', 'zip', 'model6')
from google.colab import files
files.download('model6.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
shutil.make_archive('checkpoints', 'zip', 'checkpoints')
from google.colab import files
files.download('checkpoints.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import runtime
runtime.unassign()

In [ ]:
from setfit import SetFitModel

# Specify the path to your saved checkpoint
checkpoint_path = './checkpoint-876' # Replace with the actual path to your desired checkpoint

# Load the model from the checkpoint
loaded_model = SetFitModel.from_pretrained(checkpoint_path)
loaded_model2 = SetFitModel.from_pretrained('./FIRSTSAVE')
loaded_model3 = SetFitModel.from_pretrained("malteos/scincl")


model_head.pkl not found in /content/checkpoint-876, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


In [ ]:
base_model = loaded_model.model_body
transformer = base_model._modules['0'].auto_model
embeddings = transformer.get_input_embeddings().weight
print(base_model)

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)


In [ ]:
import numpy as np
doc = "The quick brown fox jumps over the lazy dog."
embedding = loaded_model.model_body.encode(doc)
embedding2 = loaded_model2.model_body.encode(doc)
embedding3 = loaded_model3.model_body.encode(doc)

print(np.dot(embedding2, embedding3)/(np.linalg.norm(embedding2)*np.linalg.norm(embedding3)))

0.48137653


In [ ]:
# Assuming 'loaded_model' is the name of your loaded SetFit model
# The specific layer names might vary depending on the model architecture
# You might need to inspect the model's structure to find the exact layer name

# Access the underlying Sentence Transformer model
sentence_transformer_model = loaded_model.model_body

# Access the layers of the Sentence Transformer model
# The last layer before the pooling or classification head is often the one containing the final embeddings
# You might need to adjust the index based on your specific model
# For example, if the last layer is a pooling layer, you'd look at the layer before it.
try:
    # This is a common way to access the last layer before the classification head
    # in models like those used by SetFit.
    penultimate_layer = sentence_transformer_model[1]
    print(sentence_transformer_model)

    # Access the weights of the layer
    # The way to access weights depends on the layer type (e.g., Linear, Dense)
    # If it's a Linear layer, you can typically access .weight
    print("Weights of the penultimate layer:")
    print(penultimate_layer)

except IndexError:
    print("Could not access the penultimate layer. The model might have a different structure.")
except Exception as e:
    print(f"An error occurred: {e}")

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)
Weights of the penultimate layer:
Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
